# Connect scaling bounds, arithmetic intensity and collectives

Prerequisites: Runtime fractions, rates, units, arrays and reductions.

Amdahl's model gives speedup 1/[(1−p)+p/N] for a fixed workload with parallel fraction p. At p=0.8 and N=8, the ideal speedup is 3.333…, not eight. Communication, synchronization, scheduling and memory contention add costs outside this simple upper bound.

The Roofline model bounds attained compute rate by min(peak FLOP/s, bandwidth×arithmetic intensity). Multiplying 100 GB/s by 2 FLOP/byte gives 200 GFLOP/s; a 1,000 GFLOP/s compute ceiling is not attainable at that intensity. The ridge point is peak/bandwidth, here 10 FLOP/byte. Peak and bandwidth must describe the relevant memory level and precision.

The core notebook simulates partitioning integers 1..100 among ranks and summing partial results sequentially. The optional mpi_sum.py actually calls MPI Allreduce when run under mpiexec with mpi4py installed. These are separate execution statuses. A correctness result of 5,050 is not a measured parallel speedup. For performance, record machine, versions, rank placement, warm-up, repetitions and timing boundaries.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Amdahl | Serial and parallel time fractions | Scaling upper bound |
| Roofline | min(peak,B×I) | Compute or memory bound |
| Allreduce | Local sums → shared total | Communication semantics |

## Predict

Defaults yield ideal speedup 10/3, Roofline bound 200 GFLOP/s and reduction result 5,050. Change intensity from 2 to 20 and observe the compute ceiling cap at 1,000.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import systems_models as m

In [ ]:
result=m.hpc()
assert abs(result['amdahl_speedup']-10/3)<1e-12
assert result['roofline_gflops']==200
assert result['simulated_reduce']==5050
assert m.hpc(intensity=20)['roofline_gflops']==1000
print(result)

## Perturb and explain

Run the optional MPI program on 1,2,4 ranks in a suitable environment. Separate the correctness result from performance and explain why a tiny workload may slow down.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

At 2 FLOP/byte and 100 GB/s, the bandwidth bound is:

1. 200 GFLOP/s
2. 1,000 GFLOP/s regardless of intensity
3. 50 GB/s

Answer and explanations: 1: The units cancel bytes, yielding a compute rate. 2: That is the separate peak ceiling. 3: This divides incompatible quantities for the requested bound.

## Primary evidence

- [Williams et al. — Roofline](https://doi.org/10.1145/1498765.1498785)
- [MPI Forum — MPI 5.0 standard](https://www.mpi-forum.org/docs/mpi-5.0/mpi50-report.pdf)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.